In [14]:
# 1. Установка необходимых библиотек
import sys
# Resolve dependency conflicts by enforcing modern versions
!{sys.executable} -m pip uninstall -y langchain langchain-community langchain-core pydantic trl
# Remove version constraint on TRL to get latest (compatible with transformers 4.40+)
!{sys.executable} -m pip install -q "langchain>=0.2" "langchain-community>=0.2" "langchain-core>=0.2" "pydantic>=2" torch transformers peft bitsandbytes "trl" datasets accelerate diffusers rich

Found existing installation: langchain 1.2.8
Uninstalling langchain-1.2.8:
  Successfully uninstalled langchain-1.2.8
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-core 1.2.8
Uninstalling langchain-core-1.2.8:
  Successfully uninstalled langchain-core-1.2.8
Found existing installation: pydantic 2.12.5
Uninstalling pydantic-2.12.5:
  Successfully uninstalled pydantic-2.12.5
Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6


In [15]:
# 2. Импорт зависимостей и настройка окружения
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from langchain.tools import tool

# LangChain imports with error handling for environment issues
try:
    from langchain.agents import AgentExecutor, create_json_chat_agent
except ImportError:
    print("Warning: LangChain AgentExecutor/create_json_chat_agent not found. Check installation.")
    AgentExecutor = None
    create_json_chat_agent = None

from langchain_community.llms import HuggingFacePipeline
try:
    from langchain_core.prompts import ChatPromptTemplate
except ImportError:
    try:
        from langchain.prompts import ChatPromptTemplate
    except ImportError:
         print("Warning: ChatPromptTemplate not found.")
         ChatPromptTemplate = None

# Настройка логирования
logging.set_verbosity_info()

# Проверка GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device using: {device}")

Device using: cuda


In [16]:
# 3. Загрузка базовой модели и токенизатора

model_name = "NousResearch/Llama-2-7b-chat-hf" # Используем Llama-2-7b как базовую модель
new_model_name = "llama-2-7b-finetome-lora"

# Конфигурация квантования (4-bit) для экономии памяти
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Загрузка базовой модели
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

loading configuration file config.json from cache at /home/zodiac/.cache/huggingface/hub/models--NousResearch--Llama-2-7b-chat-hf/snapshots/351844e75ed0bcbbe3f10671b3c808d2b83894ee/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 4096,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 32000
}

loading weights file model.safetensors from cache at /hom

In [17]:
# 4. Подготовка и предобработка датасета

# Загрузка датасета FineTome-100k
dataset_name = "mlabonne/FineTome-100k"
dataset = load_dataset(dataset_name, split="train[:1000]") # Берем подмножество для демонстрации

print(f"Пример данных:\n{dataset[0]}")

# Функция форматирования промптов (используем ChatML/Messages format)
dataset_text_field = "formatted_text"

def format_instruction(sample):
    conversations = sample['conversations']
    messages = []
    
    for turn in conversations:
        role = "user" if turn['from'] == 'human' else "assistant"
        content = turn['value']
        messages.append({"role": role, "content": content})
            
    return {"formatted_text": messages}

# Применяем форматирование
dataset = dataset.map(format_instruction)
print(f"Форматированный пример:\n{dataset[0]['formatted_text']}")

Пример данных:
{'conversations': [{'from': 'human', 'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'}, {'f

In [18]:
# 5. Конфигурация LoRA и PEFT

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [19]:
# 6. Запуск процесса дообучения (Fine-tuning)
from trl import SFTConfig
from trl import SFTTrainer
import os

# Преобразование данных в формат ChatML
# SFTTrainer ожидает текстовое поле для обучения. Преобразуем список сообщений в строку формата ChatML.
def format_to_chatml(example):
    messages = example.get('formatted_text') 
    if not isinstance(messages, list):
         # Если formatted_text уже не список, пробуем взять conversations
         messages = example.get('conversations')
    
    # Генерация строки ChatML
    text = ""
    if isinstance(messages, list):
        for message in messages:
            role = message.get('role', 'user')
            content = message.get('content', '')
            text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    else:
        text = str(messages)
        
    return {"formatted_text": text}

print("Форматирование датасета в ChatML...")
# Принудительная обработка без кэша для гарантии корректности
dataset = dataset.map(format_to_chatml, load_from_cache_file=False)

# Очистка лишних колонок (conversation, messages и т.д.), чтобы они не мешали Trainer'у
raw_columns = [col for col in dataset.column_names if col != "formatted_text"]
dataset = dataset.remove_columns(raw_columns)

print(f"Пример данных после форматирования: {dataset[0]['formatted_text'][:100]}...")

# Настройка chat_template для токенизатора (необходимо для TRL v0.27+)
if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

# Конфигурация SFT (TRL v0.27+ использует SFTConfig)
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False, # Используйте True если GPU поддерживает (например, Ampere)
    max_grad_norm=0.3,
    max_steps=50, # Ограничение шагов для демонстрации. Для реального обучения установите -1
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    dataset_text_field="formatted_text",
    max_length=512, 
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer, # В новых версиях параметр называется processing_class
    args=sft_config,
)

# Отключение генерации карточки модели (во избежание проблем с отсутствующими шаблонами)
trainer.create_model_card = lambda *args, **kwargs: None

print("Запуск обучения...")
trainer.train()
print("Обучение завершено!")

ImportError: cannot import name 'SFTConfig' from 'trl' (/home/zodiac/stadygit/LLM-Driven-Development/.venv/lib/python3.11/site-packages/trl/__init__.py)

In [ ]:
# Настройка шаблона чата (Chat Template)
# Поскольку модель может не иметь встроенного шаблона, задаем шаблон ChatML вручную
tokenizer.chat_template = "{% if messages[0]['role'] == 'system' %}{% set loop_messages = messages[1:] %}{% set system_message = messages[0]['content'] %}{% else %}{% set loop_messages = messages %}{% set system_message = False %}{% endif %}{% if system_message %}{{ '<|im_start|>system\n' + system_message + '<|im_end|>\n' }}{% endif %}{% for message in loop_messages %}{% if message['role'] == 'user' %}{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}{% elif message['role'] == 'assistant' %}{{ '<|im_start|>assistant\n' + message['content'] + '<|im_end|>\n' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

# Убедимся, что добавлены спецтокены для ChatML, если их нет
# Llama-2 tokenizer usually usually doesn't have these by default
special_tokens_dict = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}
tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))


In [ ]:
# 7. Сохранение адаптеров модели

trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)

print("Модель (адаптеры) сохранена.")

In [ ]:
# 8. Определение пользовательских инструментов (Tools)
from langchain.tools import tool

@tool
def format_to_upper(text: str) -> str:
    """
    Converts the text to upper case.
    Use this tool when the user wants to capitalize text.
    Input: The text string to convert.
    """
    return text.upper()

@tool
def format_to_lower(text: str) -> str:
    """
    Converts the text to lower case.
    Use this tool when the user wants to lowercase text.
    Input: The text string to convert.
    """
    return text.lower()

@tool
def content_validator(text: str) -> str:
    """
    Validates the text content. Checks if the length meets the minimum requirement (5 chars).
    Returns 'Valid' or an error message.
    """
    if len(text) >= 5:
        return "Valid"
    else:
        return "Invalid: Text is too short."

tools = [format_to_upper, format_to_lower, content_validator]

In [ ]:
# 9. Интеграция модели с инструментами и тестирование
import re
import torch

# Важно: переключаем модель в режим оценки (Inference Mode)
model.eval()

# Создание пайплайна
# Явно задаем параметры генерации
gen_kwargs = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.01, # Низкая температура для стабильности работы агента
    "top_k": 50,
    "top_p": 0.95,
    "repetition_penalty": 1.05,
    "pad_token_id": tokenizer.eos_token_id
}

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    **gen_kwargs
)

# Кастомная реализация ReAct агента (устойчивая к версиям библиотек)
class SimpleReactAgent:
    def __init__(self, pipeline, tools, template):
        self.pipeline = pipeline
        self.tools = {t.name: t for t in tools}
        self.template = template
        
    def run(self, user_input, max_steps=4):
        tool_desc = "\n".join([f"{t.name}: {t.description}" for t in self.tools.values()])
        tool_names = ", ".join(self.tools.keys())
        
        # Заполнение шаблона промпта
        prompt = self.template.replace("{tools}", tool_desc).replace("{tool_names}", tool_names).replace("{input}", user_input)
        scratchpad = ""
        
        current_prompt = prompt.replace("{agent_scratchpad}", scratchpad)
        
        print(f"Вопрос пользователя: {user_input}")
        
        for step in range(max_steps):
            print(f"--- Шаг {step + 1} ---")
            
            # Генерация ответа модели
            outputs = self.pipeline(current_prompt)
            generated_text = outputs[0]['generated_text']
            
            new_text = generated_text
            
            # Обрезаем лишнее (если модель сгенерировала наблюдение за нас или начала новый вопрос)
            if "Observation:" in new_text:
                new_text = new_text.split("Observation:")[0]
            if "Question:" in new_text:
                 new_text = new_text.split("Question:")[0]
            
            print(f"Мысли модели:\n{new_text.strip()}\n")
            
            # Проверка на наличие финального ответа
            if "Final Answer:" in new_text:
                answer = new_text.split("Final Answer:")[-1].strip()
                return answer
            
            # Парсинг действия (Action)
            action_match = re.search(r"Action:\s*(.*?)\nAction Input:\s*(.*)", new_text, re.DOTALL)
            
            if action_match:
                action_name = action_match.group(1).strip()
                action_input = action_match.group(2).strip()
                
                print(f"Вызов инструмента: {action_name} с вводом: {action_input}")
                
                if action_name in self.tools:
                    try:
                        action_input = action_input.strip('"\'')
                        observation = self.tools[action_name].run(action_input)
                    except Exception as e:
                        observation = f"Ошибка: {e}"
                else:
                    observation = f"Ошибка: Инструмент {action_name} не найден."
                    
                print(f"Результат (Observation): {observation}")
                
                # Добавляем результат в историю (scratchpad) и обновляем промпт
                step_text = f"{new_text.strip()}\nObservation: {observation}\nThought:"
                current_prompt += step_text
                
            else:
                print("Действие не распознано. Остановка.")
                return new_text.strip()
                
        return "Достигнут лимит шагов."

# Промпт в стиле ReAct (Оставляем на английском для лучшего качества работы Llama-2)
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

# Инициализация и запуск
agent = SimpleReactAgent(pipe, tools, template)

try:
    print("Тестирование агента на запросе: 'Format the text hello world to upper case'")
    response = agent.run("Format the text 'hello world' to upper case.")
    print(f"\nФинальный ответ: {response}")
except Exception as e:
    print(f"Ошибка выполнения: {e}")